# MM-CLightRec v2 — MovieLens 1M Dataset

**Upgrades over v1:**
- ✅ Uses **real TMDB movie poster features** (`image_feat.npy`) instead of random noise
- ✅ Includes **Sampled Negative Evaluation** matching the base paper (MGRS-HFA) protocol
- ✅ Cold-Start Contrastive Loss (L3) enabled

### Step 1: Upload the Project & Image Features
Upload `mm_clightrec_ml1m_code.zip` **and** `image_feat.npy` to the root of this Colab session, then run the cells below.

In [ ]:
!unzip -o -q mm_clightrec_ml1m_code.zip

# Move image_feat.npy to where data_loader.py expects it
import os, shutil
os.makedirs('ml-1m', exist_ok=True)
if os.path.exists('/content/image_feat.npy'):
    shutil.copy('/content/image_feat.npy', 'ml-1m/image_feat.npy')
    print('✅ Real image features copied to ml-1m/image_feat.npy')
else:
    print('⚠️  image_feat.npy not found. Will use synthetic proxy instead.')
    print('   Upload image_feat.npy alongside the zip for better metrics.')

### Step 2: Install Required Libraries

In [ ]:
!pip install scikit-learn pandas numpy torch torchvision tqdm -q
print('All packages ready ✅')

In [ ]:
# ==============================================================================
# 💾 STEP 2.5: Mount Google Drive & Protect Your Results
# ==============================================================================
# This connects your Google Drive and forces the model to save everything 
# directly to your Drive. If Colab disconnects, you won't lose your 8-hour run!

from google.colab import drive
import os

print("[1] Mounting Google Drive...")
drive.mount('/content/drive')

print("\n[2] Setting up permanent storage folder...")
DRIVE_DIR = '/content/drive/MyDrive/MM_CLightRec_ML1M_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)

print("\n[3] Linking local results to Google Drive...")
!rm -rf results
!ln -s "{DRIVE_DIR}" results

print(f"\n✅ SUCCESS! All models, plots, and metrics will save directly to:\n   {DRIVE_DIR}")

### Step 3: Run Training on MovieLens 1M
This automatically loads `ml-1m/image_feat.npy` for real image features and trains for 300 epochs.

In [ ]:
!python main.py --dataset ml1m --epochs 300 --batch_size 4096 --include_cold_start

### Step 4: Sampled Negative Evaluation (Matches Base Paper Protocol)

The training above uses **All-Item Ranking** (rigorous). This cell also runs **Sampled Negative Evaluation** (1 pos + 99 neg per user) which matches the MGRS-HFA base paper's evaluation protocol — giving directly comparable metrics.

**No retraining needed — loads the best saved model automatically.**

In [ ]:
import torch, numpy as np, os, sys
sys.path.insert(0, '/content')

# ── Config ────────────────────────────────────────────────────────────────
N_NEGATIVE = 99   # 1 positive + 99 negatives = 100 candidates (standard protocol)
K          = 10
SEED       = 42
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load data (re-uses same pipeline so IDs match) ─────────────────────
print('[INFO] Loading ML-1M data...')
from data_loader import load_and_preprocess_ml1m
data = load_and_preprocess_ml1m()

n_users = data['n_users']
n_items = data['n_items']
feature_dim   = data['feature_dim']
modality_dims = data['modality_dims']
user_features = data['user_features'].to(device)
item_features = data['item_features'].to(device)
edge_index    = data['edge_index'].to(device)
user_modality_features = {k: v.to(device) for k, v in data['user_modality_features'].items()}
item_modality_features = {k: v.to(device) for k, v in data['item_modality_features'].items()}
user_idx  = data['user_idx']
item_idx  = data['item_idx']
test_idx  = data['test_idx']
train_idx = data['train_idx']
val_idx   = data['val_idx']

# ── Load best model ───────────────────────────────────────────────────────
from models.mm_clightrec import MM_CLightRec

model = MM_CLightRec(
    n_users=n_users, n_items=n_items,
    user_feature_dim=feature_dim, item_feature_dim=feature_dim,
    modality_dims=modality_dims,
    include_cold_start=True
).to(device)

for candidate in ['results/best_model.pth', 'results/mm_clightrec_ml1m.pth']:
    if os.path.exists(candidate):
        state = torch.load(candidate, map_location=device)
        if isinstance(state, dict) and 'model_state_dict' in state:
            model.load_state_dict(state['model_state_dict'])
        else:
            model.load_state_dict(state)
        print(f'[INFO] Loaded model from {candidate} ✅')
        break
else:
    print('[WARNING] No saved model found. Using untrained model — results will be random.')

model.eval()

# ── Compute full score matrix (n_users × n_items) ─────────────────────
print('[INFO] Computing score matrix...')
with torch.no_grad():
    score_matrix = model.get_all_scores(
        user_features, item_features, edge_index,
        user_modality_features, item_modality_features
    ).cpu().numpy()
print(f'[INFO] Score matrix ready: {score_matrix.shape}')

# ── Build lookup dicts ────────────────────────────────────────────────
def build_user_items_dict(u_idx, i_idx, interaction_idx):
    d = {}
    for idx in interaction_idx:
        u, i = int(u_idx[idx]), int(i_idx[idx])
        d.setdefault(u, []).append(i)
    return d

train_ui = build_user_items_dict(user_idx, item_idx, train_idx)
val_ui   = build_user_items_dict(user_idx, item_idx, val_idx)
test_ui  = build_user_items_dict(user_idx, item_idx, test_idx)

# ── Sampled Negative Evaluation ────────────────────────────────────────
def compute_ndcg_sampled(ranked_list, pos_item, k):
    for rank, item in enumerate(ranked_list[:k]):
        if item == pos_item:
            return 1.0 / np.log2(rank + 2)
    return 0.0

def sampled_evaluate(score_matrix, test_user_items, train_user_items,
                     n_items, n_neg=99, k=10, seed=42):
    rng = np.random.RandomState(seed)
    prec_list, rec_list, ndcg_list, f1_list = [], [], [], []

    for u, pos_items in test_user_items.items():
        if not pos_items:
            continue
        pos_item = pos_items[0]  # take first positive (standard)
        seen     = set(train_user_items.get(u, [])) | set(pos_items)

        # Sample n_neg random negatives not seen by this user
        unseen = np.array([i for i in range(n_items) if i not in seen])
        negs   = rng.choice(unseen, size=min(n_neg, len(unseen)), replace=False)
        candidates = np.concatenate([[pos_item], negs])

        # Score and rank the candidate pool
        s        = score_matrix[u][candidates]
        order    = np.argsort(-s)
        ranked   = candidates[order]

        hits = int(pos_item in ranked[:k])
        p    = hits / k
        r    = float(hits)          # recall = hits/1 (1 positive)
        nd   = compute_ndcg_sampled(ranked, pos_item, k)
        f    = 2*p*r/(p+r) if (p+r) > 0 else 0.0

        prec_list.append(p)
        rec_list.append(r)
        ndcg_list.append(nd)
        f1_list.append(f)

    return {
        f'Precision@{k}': float(np.mean(prec_list)),
        f'Recall@{k}':    float(np.mean(rec_list)),
        f'NDCG@{k}':      float(np.mean(ndcg_list)),
        f'F1@{k}':        float(np.mean(f1_list)),
        'Users evaluated': len(prec_list),
    }

print(f'\n[INFO] Running Sampled Evaluation (1 pos + {N_NEGATIVE} neg per user)...')
sampled_m = sampled_evaluate(
    score_matrix, test_ui, train_ui,
    n_items=n_items, n_neg=N_NEGATIVE, k=K, seed=SEED
)

# ── Print Results ─────────────────────────────────────────────────────
print('\n' + '='*68)
print(f'  SAMPLED NEGATIVE EVALUATION  (1 pos + {N_NEGATIVE} neg, K={K})')
print(f'  Protocol matches base paper — MGRS-HFA')
print('='*68)
for metric, value in sampled_m.items():
    if isinstance(value, float):
        print(f'  {metric:<30} {value:.4f}')
    else:
        print(f'  {metric:<30} {value}')

print('\n── Comparison Table ─────────────────────────────────────────────')
print(f'{"Metric":<25} {"Base Paper":>12} {"Ours (All-Item)":>16} {"Ours (Sampled)":>15}')
print('-'*70)
# All-item results from previous training run
all_item = {'P': 0.0442, 'R': 0.0502, 'N': 0.0580, 'F': 0.0390}
base     = {'P': 0.8269, 'R': 0.8718, 'N': 0.6844, 'F': 0.8484}
s        = sampled_m
print(f'{"Precision@10":<25} {base["P"]:>12.4f} {all_item["P"]:>16.4f} {s[f"Precision@{K}"]:>15.4f}')
print(f'{"Recall@10":<25} {base["R"]:>12.4f} {all_item["R"]:>16.4f} {s[f"Recall@{K}"]:>15.4f}')
print(f'{"NDCG@10":<25} {base["N"]:>12.4f} {all_item["N"]:>16.4f} {s[f"NDCG@{K}"]:>15.4f}')
print(f'{"F1@10":<25} {base["F"]:>12.4f} {all_item["F"]:>16.4f} {s[f"F1@{K}"]:>15.4f}')
print('='*70)
print('\nNote: All-Item Ranking is a harder, more rigorous evaluation.')
print('Sampled Evaluation is used for direct comparison with the base paper.')


### Step 5: Zip Results for Download

In [ ]:
!zip -r results_ml1m_v2.zip results/
print('Results zipped ✅ — Download results_ml1m_v2.zip from the file panel.')